In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

from ncps.torch import LTC
from ncps.wirings import AutoNCP

RAND_SEED = 5904
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)

## Load Data

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

NUM_FEATURES = train_X.shape[1]

train_surv_Y = Surv.from_arrays(train_Y[:,0], train_Y[:,1])
test_surv_Y = Surv.from_arrays(test_Y[:,0], test_Y[:,1])

## LTC LNN Model

In [ ]:
class ProbabilityPredictorLTC(nn.Module):
    def __init__(self, input_size, wiring, batch_first=True, return_sequences=False, ode_unfolds=6):
        super(ProbabilityPredictorLTC, self).__init__()
        
        self.ltc_lnn = LTC(input_size=input_size,
                            units=wiring,
                            batch_first=batch_first,
                            return_sequences=return_sequences,
                            ode_unfolds=ode_unfolds)
        
        self.fc1 = nn.Linear(wiring.output_dim, 1)

        self.sigmoid = nn.Sigmoid()
    
    def forward(self, input, timespans):

        x, _ = self.ltc_lnn(input=input, hx=None, timespans=timespans)
        x = self.fc1(x)
        x = self.sigmoid(x)

        return x

def get_ltc_model(num_inputs, num_outputs, num_neurons, network_sparsity=0.5, ode_unfolds=6):
    
    network_wiring = AutoNCP(num_neurons, num_outputs, sparsity_level=network_sparsity, seed=RAND_SEED)

    model = ProbabilityPredictorLTC(input_size=num_inputs,
                                    wiring=network_wiring,
                                    batch_first=True,
                                    return_sequences=False,
                                    ode_unfolds=ode_unfolds)
    
    return model

In [ ]:
class RelapseDataset(torch.utils.data.Dataset):
    def __init__(self, X, dt, Y):
        self.featuresX = torch.tensor(X, dtype=torch.float32)
        self.time = torch.tensor(dt, dtype=torch.float32)
        self.relapseOutcome = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.featuresX)

    def __getitem__(self, idx):
        return self.featuresX[idx], self.time[idx], self.relapseOutcome[idx]

# Stack arrays along a new dimension
def stack_array(array, num_copies):
    stack_list = [array] * num_copies
    ret = np.stack(stack_list, axis=1)
    return ret

# Expects a list of absolute times and number of neurons in the model, in this context follow up times
def get_timespans(times, num_neurons):

    # Stack time values along a new dimension
    # Adding initial time t = ~0.0 to each timespan
    ret = np.stack([np.zeros_like((times)) + 1e-5, times], axis=1)
    
    # Convert flat array into rows
    ret = np.expand_dims(ret, axis=-1)

    # Create a copy for each neuron to use in ode_solver
    ret = np.broadcast_to(ret, (ret.shape[0], ret.shape[1], num_neurons))

    return ret

def get_ltc_dataset(num_neurons):
    # Stacking features along 2nd dimension to match the number of time steps
    train_features_X = stack_array(train_X, num_copies=2)
    test_features_X = stack_array(test_X, num_copies=2)

    # Adding initial time t = ~0.0 to each timespan
    train_times_T = get_timespans(train_Y[:,1], num_neurons)
    test_times_T = get_timespans(test_Y[:,1], num_neurons)

    # Labels
    train_labels_Y = train_Y[:,0].reshape([-1, 1])
    test_labels_Y = test_Y[:,0].reshape([-1, 1])

    # print(train_features_X.shape)
    # print(train_times_T.shape)
    # print(train_labels_Y.shape)

    training_data = RelapseDataset(train_features_X, train_times_T, train_labels_Y)
    testing_data = RelapseDataset(test_features_X, test_times_T, test_labels_Y)

    trainloader = torch.utils.data.DataLoader(training_data, batch_size=16, shuffle=True)
    testloader = torch.utils.data.DataLoader(testing_data, batch_size=16, shuffle=False)

    return trainloader, testloader

In [ ]:
def get_perf_metrics(model, dataloader, device=torch.device("cpu")):
    predictions_y_hat = []
    true_y = []
    time_vals = []

    with torch.no_grad():
        model.eval()
        for data in dataloader:
            features, times, labels = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)

            output = model(input=features, timespans=times)
            
            # Convert the output from tensor to single values
            preds = output.reshape(-1).tolist()
            truths = labels.reshape(-1).tolist()
            time_t = times[:,1].T[0].tolist()

            # Append to predictions list
            predictions_y_hat = predictions_y_hat + preds
            true_y = true_y + truths
            time_vals = time_vals + time_t

    roc = roc_auc_score(true_y, predictions_y_hat)
    mse = mean_squared_error(true_y, predictions_y_hat)
    rmse = root_mean_squared_error(true_y, predictions_y_hat)
    concordance_data = concordance_index_censored(np.array(true_y).astype(bool), time_vals, predictions_y_hat)

    print("ROC: {}".format(roc))
    # print("MSE: {}".format(mse))
    print("RMSE: {}".format(rmse))
    print("C-Index: {}".format(concordance_data[0]))

    model.train()

In [ ]:
BATCH_SIZE = 16
BATCH_PRINT_STEP = 40
NUM_EPOCHS = 100


if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

num_neurons = 24
num_outputs = 8

model = get_ltc_model(NUM_FEATURES, num_outputs, num_neurons, network_sparsity=0.75)
trainloader, testloader = get_ltc_dataset(num_neurons)

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

for epoch in range(0, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        features, times, labels = data

        features = features.to(device)
        times = times.to(device)
        labels = labels.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward
        output = model(input=features, timespans=times)

        # Backward
        loss = criterion(output, labels)
        loss.backward()
        
        # Optimize
        optimizer.step()

        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / BATCH_PRINT_STEP
            epochBatchLossPrint = "Epoch: {} Batch: {} Loss: {:.5f}".format(epoch + 1, i + 1, curr_loss)
            print(epochBatchLossPrint)
            running_loss = 0.0
    

    print("Training Data:")
    get_perf_metrics(model=model, dataloader=trainloader, device=device)
    print("Testing Data:")
    get_perf_metrics(model=model, dataloader=testloader, device=device)
    

In [ ]:
if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model = model.to(device)

print("Training Data:")
get_perf_metrics(model=model, dataloader=trainloader, device=device)
print("Testing Data:")
get_perf_metrics(model=model, dataloader=testloader, device=device)


In [ ]:
pred_list = list()

for test_index in range(0, test_Y.shape[0]):

    # print(train_features_X[0].shape)
    # print(train_times_T[0].shape)

    # Convert a single test case to the right shape [Batch, Time, Neuron]
    # In the ode solver, the time is used for calculations in each neuron.
    # The calculations are performed elementwise so 
    # Create list of time points
    # timespans_list = [1e-5, test_Y[:,1][test_index], 3, 5, 6]

    timespans_list = [1.0] * 59

    # timespans_list[0] = 1e-5

    # Convert to numpy array
    single_test_t = np.stack(timespans_list)

    single_test_t = np.expand_dims(single_test_t, axis=-1)
    single_test_t = np.broadcast_to(single_test_t, (single_test_t.shape[0], num_neurons))
    single_test_t = torch.tensor(single_test_t).float().unsqueeze(0)
    # print(single_test_t.shape)

    # print(test_Y[:,0][test_index])

    # Convert a single test case to the right shape [Batch, Vector, Feature]
    single_feature_vector = test_X[test_index]

    # Make copies of the feature vector to match the number of prediction time points
    copied_vectors = [test_X[test_index]] * len(timespans_list)

    single_test_X = np.stack(copied_vectors, axis=0)
    # print(single_test_X.shape)
    single_test_X = torch.tensor(single_test_X).float().unsqueeze(0)
    # print(single_test_X)

    with torch.no_grad():
        model.eval()
        model.to("cpu")

        model.ltc_lnn.return_sequences = True
        pred = model(input=single_test_X, timespans=single_test_t)
        model.ltc_lnn.return_sequences = False
        
        pred_list.append(pred)
        # print(pred.reshape(-1).numpy().shape)


In [ ]:
# from sksurv.metrics import integrated_brier_score

# print(torch.tensor(pred_list))

# times = np.arange(1, 60)

# # integrated_brier_score(train_surv_Y, test_surv_Y, preds, times)


In [ ]:
# x_values = np.linspace(0, 60, 120)

# predictions = pred.reshape(-1)
# pred_probs = np.cumsum(1 - predictions)
# surv_func = np.exp(-pred_probs)
# surv_func = np.insert(surv_func, 0, 1.0)
# plot_times = np.insert(times, 0, 0)

# # print(x_values)
# plt.figure()
# plt.step(plot_times, surv_func, where="post")
# # plt.plot(x_values, surv_func)
# plt.show()